In [1]:
import os, time, warnings

import numpy as np
import pandas as pd

import albumentations as A
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torch.optim as optim

from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")



In [2]:
DIR_INPUT = "/kaggle/input/plant-pathology-2020-fgvc7"
DIR_WORK = "/kaggle/working"

SEED = 42
N_FOLDS = 5
N_EPOCHS = 6  # increased epochs for better learning
BATCH_SIZE = 16
IMAGE_SIZE = (273, 409)  # (height, width)

torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device




device(type='cuda')

In [3]:
class PlantDataset(Dataset):
    def __init__(self, df, transforms=None, test_set=False):
        self.df = df.reset_index(drop=True)
        self.test_set = test_set
        self.transforms = transforms or A.Compose(
            [A.Normalize(p=1.0), A.pytorch.transforms.ToTensorV2(p=1.0)]
        )

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(DIR_INPUT, "images", f"{row['image_id']}.jpg")
        image = cv2.imread(img_path, cv2.IMREAD_COLOR)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        image = cv2.resize(image, IMAGE_SIZE)

        transformed = self.transforms(image=image)
        image = transformed["image"]

        if self.test_set:
            return image
        else:
            labels = row[
                ["healthy", "multiple_diseases", "rust", "scab"]
            ].values.astype(np.float32)
            labels = torch.from_numpy(labels)  # shape: (4,)
            return image, labels




In [4]:
class PlantModel(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.backbone = torchvision.models.resnet50(pretrained=True)
        in_features = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity()  # remove original head
        self.logit = nn.Linear(in_features, num_classes)

    def forward(self, x):
        x = self.backbone(x)  # shape: (B, F)
        x = F.dropout(x, 0.25, self.training)
        x = self.logit(x)  # shape: (B, 4)
        return x




In [5]:
transforms_valid = A.Compose(
    [A.Normalize(p=1.0), A.pytorch.transforms.ToTensorV2(p=1.0)]
)

test_df = pd.read_csv(os.path.join(DIR_INPUT, "test.csv"))
dataset_test = PlantDataset(df=test_df, test_set=True, transforms=transforms_valid)
testloader = DataLoader(
    dataset_test, batch_size=BATCH_SIZE, shuffle=False, num_workers=4
)



In [6]:
train_df = pd.read_csv(os.path.join(DIR_INPUT, "train.csv"))

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

train_transforms = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.Resize(height=IMAGE_SIZE[0], width=IMAGE_SIZE[1]),  # ensure consistent size
        A.Normalize(p=1.0),
        A.pytorch.transforms.ToTensorV2(p=1.0),
    ]
)

for fold, (train_idx, val_idx) in enumerate(kf.split(train_df)):
    print(f"\n=== Fold {fold} ===")
    df_train = train_df.iloc[train_idx].reset_index(drop=True)
    df_val = train_df.iloc[val_idx].reset_index(drop=True)

    train_ds = PlantDataset(df_train, transforms=train_transforms, test_set=False)
    val_ds = PlantDataset(df_val, transforms=transforms_valid, test_set=False)

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4
    )
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

    model = PlantModel().to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    best_val_auc = 0.0
    for epoch in range(N_EPOCHS):
        model.train()
        epoch_loss = 0.0
        for images, targets in train_loader:
            images = images.to(device)
            targets = targets.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * images.size(0)

        epoch_loss /= len(train_loader.dataset)

        model.eval()
        all_targets = []
        all_preds = []
        with torch.no_grad():
            for images, targets in val_loader:
                images = images.to(device)
                preds = torch.sigmoid(model(images)).cpu().numpy()
                all_preds.append(preds)
                all_targets.append(targets.numpy())
        val_preds = np.concatenate(all_preds)
        val_targets = np.concatenate(all_targets)

        val_auc = np.mean(
            [roc_auc_score(val_targets[:, i], val_preds[:, i]) for i in range(4)]
        )
        print(
            f"Epoch {epoch+1}/{N_EPOCHS} - loss: {epoch_loss:.4f} - val_auc: {val_auc:.4f}"
        )

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            torch.save(model.state_dict(), os.path.join(DIR_WORK, f"modelF{fold}.pth"))

    print(f"Best val AUC for fold {fold}: {best_val_auc:.4f}")





=== Fold 0 ===


Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|██████████| 97.8M/97.8M [00:01<00:00, 99.8MB/s]


Epoch 1/6 - loss: 0.2635 - val_auc: 0.9767
Epoch 2/6 - loss: 0.1132 - val_auc: 0.9567
Epoch 3/6 - loss: 0.0952 - val_auc: 0.9673
Epoch 4/6 - loss: 0.0849 - val_auc: 0.9536
Epoch 5/6 - loss: 0.0632 - val_auc: 0.9644
Epoch 6/6 - loss: 0.0652 - val_auc: 0.9533
Best val AUC for fold 0: 0.9767

=== Fold 1 ===
Epoch 1/6 - loss: 0.2355 - val_auc: 0.9551
Epoch 2/6 - loss: 0.1133 - val_auc: 0.9372
Epoch 3/6 - loss: 0.0921 - val_auc: 0.9514
Epoch 4/6 - loss: 0.0815 - val_auc: 0.9474
Epoch 5/6 - loss: 0.0664 - val_auc: 0.9644
Epoch 6/6 - loss: 0.0581 - val_auc: 0.9608
Best val AUC for fold 1: 0.9644

=== Fold 2 ===
Epoch 1/6 - loss: 0.2539 - val_auc: 0.9620
Epoch 2/6 - loss: 0.1165 - val_auc: 0.9467
Epoch 3/6 - loss: 0.0926 - val_auc: 0.9513
Epoch 4/6 - loss: 0.0875 - val_auc: 0.9582
Epoch 5/6 - loss: 0.0670 - val_auc: 0.9782
Epoch 6/6 - loss: 0.0494 - val_auc: 0.9758
Best val AUC for fold 2: 0.9782

=== Fold 3 ===
Epoch 1/6 - loss: 0.2495 - val_auc: 0.9331
Epoch 2/6 - loss: 0.1304 - val_auc: 0.9

In [7]:
def test_model(model_name, testloader):
    """
    Load a trained model if it exists in the working directory,
    otherwise look in the original input path, otherwise fall back
    to an untrained model (so inference never crashes).
    """
    possible_paths = [
        os.path.join(DIR_WORK, f"{model_name}.pth"),
        f"/kaggle/input/plant-pathology-2020-training/{model_name}.pth",
    ]
    model = PlantModel()
    loaded = False
    for p in possible_paths:
        if os.path.exists(p):
            state = torch.load(p, map_location=device)
            model.load_state_dict(state)
            loaded = True
            print(f"Loaded model weights from {p}")
            break
    if not loaded:
        print(f"Warning: No saved model for {model_name}. Using untrained model.")
    model = model.to(device)
    model.eval()

    probs = []
    with torch.no_grad():
        for images in tqdm(testloader, leave=False):
            logits = model(images.to(device))
            prob = torch.sigmoid(logits).cpu().numpy()  # multi‑label sigmoid
            probs.append(prob)
    probs = np.concatenate(probs, axis=0)  # shape: (samples, 4)
    return probs




In [8]:
test_probs = []
start = time.perf_counter()
for i_fold in range(N_FOLDS):
    fold_probs = test_model(f"modelF{i_fold}", testloader)
    test_probs.append(fold_probs)
print(f"Finished Inference in {(time.perf_counter() - start):.2f} seconds")



Loaded model weights from /kaggle/working/modelF0.pth


  0%|          | 0/12 [00:00<?, ?it/s]

Loaded model weights from /kaggle/working/modelF1.pth


  0%|          | 0/12 [00:00<?, ?it/s]

Loaded model weights from /kaggle/working/modelF2.pth


...

Loaded model weights from /kaggle/working/modelF3.pth


  0%|          | 0/12 [00:00<?, ?it/s]

Loaded model weights from /kaggle/working/modelF4.pth


  0%|          | 0/12 [00:00<?, ?it/s]

Finished Inference in 6.45 seconds


In [9]:
test_probs_stack = np.stack(test_probs, axis=0)  # (folds, samples, 4)
test_probs_mean = test_probs_stack.mean(axis=0)  # (samples, 4)



In [10]:
submission_df = pd.read_csv(os.path.join(DIR_INPUT, "sample_submission.csv"))
submission_df[["healthy", "multiple_diseases", "rust", "scab"]] = test_probs_mean
submission_path = os.path.join(DIR_WORK, "submission.csv")
submission_df.to_csv(submission_path, index=False)
print(f"Submission written to {submission_path}")



Submission written to /kaggle/working/submission.csv


In [11]:
submission_df.head()

,image_id,healthy,multiple_diseases,rust,scab
0,Test_0,0.007389,0.057647,0.005683,0.969720
1,Test_1,0.020504,0.069612,0.005870,0.940832
2,Test_2,0.017127,0.011168,0.972071,0.009967
3,Test_3,0.000627,0.005229,0.999322,0.000746
4,Test_4,0.000205,0.003779,0.999808,0.000928
